In [ ]:
#Rank-based linear factor model for SP500
import pandas as pd
import yfinance as yf
import requests

#Getting top 500 companies from wiki
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

html = requests.get(url, headers=headers).text

sp500 = pd.read_html(html)[0]
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()  # BRK.B -> BRK-B for yfinance

data = [] 

#Storing the companies' data in a list
for t in tickers:
    try:
        info = yf.Ticker(t).info
        data.append({
            "ticker": t,
            "earnings_yield": info.get("trailingEps", None) / info.get("priceToBook", None),
            "roc": info.get("returnOnAssets", None),
            "volatility": info.get("beta", None)
        })
    except:
        pass

#Ranking the data
df = pd.DataFrame(data).dropna()
df["rank_earnings_yield"] = df["earnings_yield"].rank(ascending=False)
df["rank_roc"] = df["roc"].rank(ascending=False)
df["rank_volatility"] = df["volatility"].rank(ascending=True)

#Creating the ranking values for the algo
signs = {
    "rank_earnings_yield": +1,
    "rank_roc": +1,
    "rank_volatility": -1
}

#Calculating the final Ranks for the companies
df["final_rank"] = sum(signs[col] * df[col] for col in signs)

df = df.sort_values("final_rank", ascending=False)

top_30 = df.head(30)
print(top_30[["ticker", "final_rank"]])


C:\Users\suley\AppData\Local\Temp\ipykernel_624\1170397464.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(html)[0]
Exception ignored from cffi callback <function buffer_callback at 0x00000172A7DE0180>:
Traceback (most recent call last):
  File "c:\Users\suley\AppData\Local\Programs\Python\Python313\Lib\site-packages\curl_cffi\curl.py", line 68, in buffer_callback
    @ffi.def_extern()
KeyboardInterrupt: 


Hypothesis Testing Steps

1. Based on a backtest on some finite sample of data, we compute a certain statistical measure called the test statistic. For concreteness, lets say the test statistic is the average daily return of a trading strategy in that period.

2. We suppose that the true average daily return based on an infinite data set is actually zero. This supposition is called the null hypothesis.

3. We suppose that the probability distribution of daily returns is known. The probability distribution has a zero mean, based on the null hypothesis. We describe later how we determine this probability distribution.

4. Based on this null hypothesis probability distribution, we compute the probability p that the average daily returns will be at least as large as the observered value in the backtest. This probability p is called the p-value and if it is very small (smaller than 0.01), that means we can "reject the null hypothesis", and conclude that the backtested average daily return is statistically significant.

## Regime Identification: Mean Reversion vs Trend

Before applying any trading strategy, it is essential to understand the statistical behavior of a price series.

---

### 1. Augmented Dickey–Fuller (ADF) Test
The ADF test checks whether a price series contains a unit root, which would indicate random-walk behavior. This test is based on the regression: Δy_t = α + λ y_{t-1} + ε_t

---

The null hypothesis is \( \lambda = 0 \), meaning that price changes are independent of the current price level. Rejecting this hypothesis suggests the series is **stationary**, which is a necessary condition for mean reversion.

In practice, individual stocks often fail the ADF test due to structural changes, growth, and regime shifts. Therefore, the ADF test is used here primarily as a **statistical reference**, not as a strict trading filter.

### 2. Lambda (λ) and Half-Life of Mean Reversion
Even when the ADF test does not reject the random-walk hypothesis, the estimated value of \( \lambda \) still provides valuable economic insight. The coefficient \( \lambda \) measures how strongly price changes respond to deviations from the mean:

- \( \lambda < 0 \): mean-reverting behavior  
- \( \lambda > 0 \): trending behavior  

Interpreting the regression as an Ornstein–Uhlenbeck process allows us to compute the **half-life of mean reversion**:

\[
\text{Half-life} = \frac{-\ln(2)}{\lambda}
\]

The half-life represents the expected time for a price deviation to decay by 50%. Short half-lives indicate strong and potentially tradable mean reversion, while long half-lives suggest that mean-reversion strategies may be impractical.


### 3. Hurst Exponent
The Hurst exponent measures how the variance of price changes scales with time:

\[
\mathrm{Var}[z(t+\tau) - z(t)] \sim \tau^{2H}
\]

Interpretation:
- \( H = 0.5 \): random walk  
- \( H < 0.5 \): mean reversion  
- \( H > 0.5 \): trending behavior  

Unlike the ADF test, the Hurst exponent captures **long-range dependence** and provides a complementary, scale-based view of market behavior.

### 4. Strategy Interpretation
No single test is decisive on its own. Instead, these diagnostics should be interpreted together:

- **ADF**: statistical evidence against random walk (strict but informative)
- **λ and half-life**: economic usefulness and trading time scale
- **Hurst exponent**: persistence vs anti-persistence across horizons

If λ is negative with a reasonably short half-life and the Hurst exponent is below 0.5, a **mean-reversion strategy** may be appropriate. If λ is positive or the Hurst exponent is above 0.5, a **trend-following strategy** is more suitable. When signals are weak or contradictory, the asset may be close to a random walk, and caution is warranted.

---

### Key Takeaway
The goal of these tests is not to “prove” a strategy, but to **avoid applying the wrong strategy to the wrong market regime**. Understanding regime behavior is a critical first step in systematic trading and helps guide both strategy selection and parameter choices.

In [4]:
#Running ADF, Hurst, and Mean-life test to determine what strategy
#to decide between Mean Reversion vs Trend strategies 

import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

ticker = "AAPL"
prices = yf.download(ticker,start="2018-01-01", auto_adjust=True, progress=False)["Close"].dropna()
logp = np.log(prices)

print(f"{ticker} data points:", len(logp))

#ADF TEST
# PURPOSE:
# The ADF test checks whether the price series behaves like
# a RANDOM WALK (unit root) or is STATIONARY.
#
# Model tested (simplified):
# Δy_t = λ y_{t-1} + μ + ε_t
#
# Null hypothesis (H0):
#   λ = 0  → random walk (no mean reversion)
#
# Alternative hypothesis (H1):
#   λ < 0 → mean reverting
#
# We test this using the t-statistic of λ against
# Dickey-Fuller critical values.

adf_stat, adf_p, adf_lags, adf_nobs, adf_crit = adfuller(
    logp, maxlag=1, regression="c", autolag=None
)

print("\n--- ADF Test on log(price) ---")
print("ADF statistic:", adf_stat)
print("p-value:", adf_p)
print("lags used:", adf_lags)
print("critical values:", adf_crit)

# INTERPRETATION:
# If ADF statistic < critical value → reject random walk
# In practice, single stocks usually FAIL ADF

#Lambda + Half-life calculation
# PURPOSE:
# Even if ADF fails, traders still want to know:
# "How fast does the price mean-revert?"
#
# We estimate λ directly using the regression:
#
# Δy_t = α + λ y_{t-1} + ε_t
#
# λ < 0 → mean reversion
# λ > 0 → trending behavior
#
# From Ornstein-Uhlenbeck theory, the HALF-LIFE of mean
# reversion is:
#
#   half_life = -ln(2) / λ

# First difference: Δy_t = y_t - y_{t-1}
dy = logp.diff().dropna()
y_lag = logp.shift(1).dropna()
dy = dy.loc[y_lag.index]

X = sm.add_constant(y_lag)
ols = sm.OLS(dy,X).fit()

lambda_hat = ols.params[1]
lambda_se = ols.bse[1]
lambda_t = lambda_hat / lambda_se

half_life = (-np.log(2) / lambda_hat) if lambda_hat < 0 else np.inf

print("\n--- Lambda (λ) + Half-life ---")
print("lambda_hat:", lambda_hat)
print("SE(lambda):", lambda_se)
print("t-stat (lambda/SE):", lambda_t)
print("half_life (days):", half_life)

# INTERPRETATION:
# λ > 0        → trending (momentum strategies)
# λ < 0        → mean reverting
# half-life:
#   < 20 days  → strong mean reversion
#   20–50 days → weak but tradable
#   > 100 days → impractical

#Hurst exponent
# PURPOSE:
# Measures how variance scales with time:
#
# Var[z(t+τ) - z(t)] ~ τ^(2H)
#
# H = 0.5 → random walk
# H < 0.5 → mean reversion
# H > 0.5 → trending
#
# This is a COMPLEMENT to ADF (not a replacement).
def hurst_exponent(x, max_lag=100):
    x = np.asarray(x).reshape(-1)  # force 1D, avoids (N,1) shape issues too
    max_lag = min(max_lag, len(x) - 2)

    lags = range(2, max_lag + 1)

    # RMS of lagged differences: sqrt(E[(x_t - x_{t-lag})^2])
    tau = [np.sqrt(np.mean((x[lag:] - x[:-lag])**2)) for lag in lags]

    # slope of log(tau) vs log(lag) is H
    poly = np.polyfit(np.log(list(lags)), np.log(tau), 1)
    return poly[0]


H = hurst_exponent(logp.values, max_lag=100)
print("\n--- Hurst Exponent ---")
print("H:", H)

# INTERPRETATION:
# H < 0.5 → mean reverting
# H ≈ 0.5 → random walk
# H > 0.5 → trending


print("\n--- Quick Interpretation ---")
if lambda_hat > 0:
    print("λ > 0 → trending bias (momentum/trend-following more appropriate).")
elif np.isfinite(half_life) and half_life < 30:
    print("λ < 0 with half-life < ~30 days → mean reversion may be tradable.")
else:
    print("λ ≤ 0 but half-life is long or λ ~ 0 → weak mean reversion / near random walk.")
print("H < 0.5 suggests mean reversion; H > 0.5 suggests trending; H ≈ 0.5 suggests random walk.")
print("ADF p-value small (e.g., <0.05) would support stationarity, but stocks often fail ADF.")

AAPL data points: 2002

--- ADF Test on log(price) ---
ADF statistic: -1.054543070964476
p-value: 0.732855215205451
lags used: 1
critical values: {'1%': np.float64(-3.433623856429125), '5%': np.float64(-2.862986213505), '10%': np.float64(-2.56753990225)}

--- Lambda (λ) + Half-life ---
lambda_hat: -0.0007767551229737242
SE(lambda): 0.0007219547349205086
t-stat (lambda/SE): -1.0759055733033587
half_life (days): 892.3625478082529

--- Hurst Exponent ---
H: 0.5264771757759854

--- Quick Interpretation ---
λ ≤ 0 but half-life is long or λ ~ 0 → weak mean reversion / near random walk.
H < 0.5 suggests mean reversion; H > 0.5 suggests trending; H ≈ 0.5 suggests random walk.
ADF p-value small (e.g., <0.05) would support stationarity, but stocks often fail ADF.


C:\Users\suley\AppData\Local\Temp\ipykernel_12748\1280639043.py:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda_hat = ols.params[1]
C:\Users\suley\AppData\Local\Temp\ipykernel_12748\1280639043.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda_se = ols.bse[1]


## Cointegration

Finding a linear combination of non-stationary price series that is stationary, and then designing a trading strategy that exploits the mean-reverting behavior of that combination.

---

### Cointegrated Augmented Dickey-Fuller (CADF) Test 

A two-step procedure used to test for cointegration between two non-stationary time series. First, one price series is regressed on the other to estimate a hedge ratio and form a spread. Second, and Augmented Dickey-Fuller test is applied to this residual to determine whether the spread is stationary. If the residual is stationary, the two series are said to be cointegrated. A key limitation of the CADF test is that it is order-dependent-regressing Y on X may yield on a different result than regressing X on Y-and it can identify only one cointegrating relationship. For this reason, CASD is most commonly used in pairs trading and simple relative-value strategies.

---

### Johansen Test

A multivariate method for detecting cointegration among two or more non-stationary time series. It is based on a vector autoregression (VAR) framework and treats all variables symmetrically, making it order-independent. Rather than testing a single spread, the Johansen test determines the number of cointegrating relationships and produces the coressponding hedge ratios as eigenvectors. This allows multiple independent stationary portfolios to be identified from the same set of assets. Because of its ability to handle higher-dimensional systems and uncover all cointegration relations at once, the Johasen test is widely used in multi-asset statistical arbitrage and portfolio construction. 

In [6]:
#CADF
# Many price series (and log prices) behave like random walks (non-stationary).
# If X_t and Y_t are *cointegrated*, then there exists a hedge ratio beta such that:
#
#     spread_t = Y_t - (alpha + beta * X_t)
#
# is *stationary* (mean-reverting).
#
# Engle–Granger / CADF is:
#   (1) Estimate alpha, beta via OLS regression
#   (2) Test the residual spread for stationarity using ADF
#
# If the spread is stationary, that’s evidence of cointegration.
x_ticker = "KO"
y_ticker = "PEP"

start = "2018-01-01"
px = yf.download([x_ticker,y_ticker], start=start, auto_adjust=True,progress=False)["Close"].dropna()

logX = np.log(px[x_ticker])
logY = np.log(px[y_ticker])

# 2) OLS regression: logy = alpha + beta*logx + error
# ------------------------------------------------------------
# OLS = Ordinary Least Squares:
# It chooses alpha and beta to minimize the SUM of squared residuals:
#
#     minimize over (alpha, beta):  Σ_t (logy_t - (alpha + beta*logx_t))^2
#
# The resulting fitted line gives:
#   - beta: slope (interpreted as hedge ratio)
#   - alpha: intercept (constant offset between the two series)

X = sm.add_constant(logX)
# add_constant adds a column of 1s so the regression can estimate alpha (intercept).
# Without it, you'd be forcing alpha=0 (line through origin), often unrealistic.

model = sm.OLS(logY,X).fit()

alpha,beta = model.params["const"], model.params[x_ticker]

print("Regression (logy ~ alpha + beta*logx)")
print(f"alpha = {alpha:.6f}")
print(f"beta  = {beta:.6f}")

spread = logY - (alpha + beta * logX)

adf_stat, p_value, used_lags, nobs, crit_vals, icbest = adfuller(
    spread.dropna(),
    regression="c",
    autolag="AIC"
)

print("\nADF test on spread (residuals)")
print(f"ADF statistic = {adf_stat:.4f}")
print(f"p-value       = {p_value:.4f}")
print(f"lags used     = {used_lags}")
print("critical values:", crit_vals)

# 5) Interpreting ADF outputs
# ------------------------------------------------------------
# ADF statistic:
# - More negative => stronger evidence against unit root (more evidence of stationarity)
#
# Critical values:
# - thresholds for the ADF stat at common significance levels (1%, 5%, 10%)
# - If ADF stat is LESS than (more negative than) the 5% critical value,
#   reject H0 at 5% significance.
#
# p-value:
# - another way to decide rejection
# - p-value < 0.05 -> reject H0 (evidence spread is stationary)
#
# Lags used:
# - ADF includes lagged Δspread terms to account for autocorrelation
# - too few lags -> test may be invalid (residual autocorrelation)
# - too many lags -> lose power / fewer effective observations


Regression (logy ~ alpha + beta*logx)
alpha = 1.456845
beta  = 0.871173

ADF test on spread (residuals)
ADF statistic = -1.5291
p-value       = 0.5191
lags used     = 2
critical values: {'1%': np.float64(-3.4336254962865045), '5%': np.float64(-2.862986937508278), '10%': np.float64(-2.567540287745173)}


In [ ]:
#JohansenTest
# Johansen tests cointegration among 2+ non-stationary time series.
# It answers:
#   - How many cointegrating relationships exist? (the "rank")
#   - What are the hedge ratios (cointegrating vectors) for each relationship?
#
# Compared to CADF:
# - Works for 2 or more series
# - Order-independent
# - Can find multiple independent cointegrating vectors

from statsmodels.tsa.vector_ar.vecm import coint_johansen

tickers = ["KO", "PEP", "PG"]
start = "2018-01-01"

px = yf.download(tickers, start=start, auto_adjust=True, progress=False)["Close"].dropna()

logpx = np.log(px)

# coint_johansen expects a 2D array / DataFrame with shape (T, n_assets).
#
# det_order controls deterministic terms:
#   -1 : no deterministic terms
#    0 : constant term in cointegration relationship (most common for spreads)
#    1 : linear trend, etc.
#
# k_ar_diff is the number of lagged differences included (VAR lags in differenced form).
# A small number like 1 or 2 is common to start with, but it can be tuned.
det_order = 0
k_ar_diff = 1

jres = coint_johansen(logpx, det_order=det_order, k_ar_diff=k_ar_diff)

# jres.lr1  = trace statistics for hypotheses r <= 0, r <= 1, ...
# jres.cvt  = critical values for trace statistic at [90%, 95%, 99%]
#
# Rule: reject H0 (r <= r0) if trace_stat > critical_value
trace_stats = jres.lr1
trace_cv = jres.cvt

print("Johansen Trace Test")
for i in range(len(trace_stats)):
    print(
        f"H0: rank <= {i} | trace stat = {trace_stats[i]:.3f} | "
        f"crit(95%) = {trace_cv[i, 1]:.3f}"
    )

# Determine rank at 95% level:
rank_95 = 0
for i in range(len(trace_stats)):
    if trace_stats[i] > trace_cv[i,1]:
        rank_95 = i +1
print(f"\nEstimated cointegration rank at 95% (trace test): {rank_95}")

# jres.evec columns are cointegrating vectors (beta vectors).
# Each vector gives weights w such that:
#   spread_t = w1*logP1_t + w2*logP2_t + ... + wn*logPn_t
# is (approximately) stationary.
#
# The first vector (column 0) is often the "strongest" relation.
evec = pd.DataFrame(jres.evec, index=tickers)
print("\nCointegrating vectors (hedge ratios) — columns are different relations:")
print(evec)

# Build the first stationary portfolio (spread) using the first eigenvector:
w = evec.iloc[:,0]

# Optional normalization: scale weights so the first asset weight is 1
# (This makes the hedge ratio easier to interpret.)
w_norm = w / w.iloc[0]

spread = logpx @ w_norm  # matrix multiply: (T x n) @ (n x 1) -> (T x 1)

print("\nNormalized weights for the first cointegrating relation:")
print(w_norm)

print("\nSpread (first 5 values):")
print(spread.head())

# HOW TO READ THE JOHANSEN TRACE TEST:
# For each hypothesis H0: rank <= r, compare:
#     trace_stat  vs  critical_value (95%)
#
# Decision rule:
# - If trace_stat > crit(95%): reject H0  -> rank is at least r+1
# - If trace_stat <= crit(95%): fail to reject H0 -> STOP
#
# The cointegration rank is the largest r for which H0 is rejected.
# If we fail at r = 0 (i.e., rank <= 0 not rejected), then rank = 0,
# meaning there is NO statistically significant cointegration.
# In that case, any eigenvectors/spreads returned are NOT tradable
# as mean-reverting strategies.

#pg 82

Johansen Trace Test
H0: rank <= 0 | trace stat = 13.481 | crit(95%) = 29.796
H0: rank <= 1 | trace stat = 6.222 | crit(95%) = 15.494
H0: rank <= 2 | trace stat = 2.651 | crit(95%) = 3.841

Estimated cointegration rank at 95% (trace test): 0

Cointegrating vectors (hedge ratios) — columns are different relations:
             0         1         2
KO    7.084959 -7.998449  5.193850
PEP   8.157424  4.528929 -7.435521
PG  -11.658311 -0.061131 -0.841092

Normalized weights for the first cointegrating relation:
KO     1.000000
PEP    1.151372
PG    -1.645501
Name: 0, dtype: float64

Spread (first 5 values):
Date
2018-01-02    1.713198
2018-01-03    1.709971
2018-01-04    1.718024
2018-01-05    1.720028
2018-01-08    1.703257
dtype: float64


In [ ]:
#Linear Mean Reverting Strategy on GLD vs USO 
#   1) Build a "signal" that should mean-revert (spread / log-spread / ratio)
#   2) Convert that signal into a z-score using rolling mean/std
#   3) Take position size = -zscore  (if signal is high, short it; if low, long it)
#   4) Compute daily PnL using yesterday’s positions and today’s price changes
#
# We do this for 3 signals to compare:
#   A) Price spread:     s_t = USO_t - beta_t * GLD_t
#   B) Log price spread: s_t = log(USO_t) - beta_t * log(GLD_t)
#   C) Ratio:            s_t = USO_t / GLD_t
#
import yfinance as yf
import numpy as np
import pandas as pd
import statsmodels.api as sm

tickers = ["GLD", "USO"]
prices = yf.download(tickers,start="2020-01-01", auto_adjust=True, progress=False)["Close"].dropna()

lookback = 20

x = prices["GLD"]
y = prices["USO"]

def zscore(s, L):
    #z_t = (s_t - MA_L(s)) / STD_L(s)
    #This measures how many standard deviations the signal is away from its recent mean.
    return (s - s.rolling(L).mean()) / s.rolling(L).std()

def rolling_beta(x_series, y_series,L):
    """
    Each day t:
      Fit regression on last L days:
        y = c + beta * x + noise
      Store beta as hedge ratio for day t.

    This is how the book dynamically updates hedgeRatio(t).
    """
    beta = pd.Series(index=x_series.index, dtype=float)

    for t in range(L -1, len(x_series)):
         # Select rolling window [t-L+1 ... t]
        xs = x_series.iloc[t - L + 1 : t + 1]
        ys = y_series.iloc[t - L + 1 : t + 1]

        X = sm.add_constant(xs.values)
        fit = sm.OLS(ys.values,X).fit()

        beta.iloc[t] = fit.params[1]

    return beta
# 4) Compute rolling hedge ratios for:
#    - price spread (regress y on x)
#    - log spread (regress log(y) on log(x))
beta_price = rolling_beta(x,y,lookback)
beta_log = rolling_beta(np.log(x),np.log(y),lookback)

# 5) Build the 3 mean-reverting signals (the "thing" we trade)
# ------------------------------------------------------------
# A) Price spread (book calls this the spread/portfolio value):
#    s_t = y_t - beta_t * x_t
spread_price = y - beta_price * x

# B) Log price spread:
#    s_t = log(y_t) - beta_t * log(x_t)
spread_log = np.log(y) - beta_log * np.log(x)

# C) Ratio:
#    s_t = y_t / x_t
ratio = y / x

# 6) Convert signals into position sizes using negative z-score
# ------------------------------------------------------------
# Book rule:
#    numUnits_t = -Zscore(signal_t)
# Interpretation:
#   - If signal is HIGH vs recent mean (z positive), numUnits becomes negative -> short signal
#   - If signal is LOW  vs recent mean (z negative), numUnits becomes positive -> long signal

u_price = -zscore(spread_price,lookback)
u_log = -zscore(spread_log,lookback)
u_ratio = -zscore(ratio,lookback)

# Translate "numUnits" into dollar positions in each ETF
# ------------------------------------------------------------
# For spread strategies:
#   The book’s positions are market values (dollars) in each ETF.
#   We mimic that:
#     GLD leg dollar exposure  ~ u_t * beta_t * GLD_price
#     USO leg dollar exposure  ~ u_t * (-1)   * USO_price
#
# Why multiply by price?
#   Because we want positions measured in dollars, and later PnL uses percent returns.
pos_price = pd.DataFrame({
    "GLD": u_price * beta_price * x,
    "USO": u_price * (-1.0) * y
})

pos_log = pd.DataFrame({
    "GLD": u_log * beta_log * x,
    "USO": u_log * (-1.0) * y
})

# For ratio strategy (book uses [-1, +1] with equal dollars long/short):
pos_ratio = pd.DataFrame({
    "GLD": u_ratio * (-1.0) * x,
    "USO": u_ratio * ( 1.0) * y
})

# 8) Compute daily strategy returns (PnL / gross exposure)
# ------------------------------------------------------------
# First compute each ETF's daily percent return:
#   r_t = (P_t - P_{t-1}) / P_{t-1}
asset_ret = prices.pct_change()

def strategy_returns(pos_dollars, asset_ret):
    """
    This matches the book:
      pnl_t = sum( pos_{t-1} * return_t )
      gross_t = sum( abs(pos_{t-1}) )
      strat_ret_t = pnl_t / gross_t
    Using pos_{t-1} avoids "lookahead" (you can only trade using yesterday’s signal).
    """
    pnl = (pos_dollars.shift(1) * asset_ret).sum(axis=1)
    gross = pos_dollars.shift(1).abs().sum(axis=1)
    return pnl / gross

r_price = strategy_returns(pos_price, asset_ret)
r_log   = strategy_returns(pos_log,   asset_ret)
r_ratio = strategy_returns(pos_ratio, asset_ret)

def apr_sharpe(r, ann=252):
    """
    APR (approx) = mean daily return * 252
    Sharpe (annualized) = sqrt(252) * mean / std
    """
    r = r.dropna()
    apr = r.mean() * ann
    sharpe = np.sqrt(ann) * r.mean() / r.std(ddof=1)
    return apr, sharpe

for name, r in [("Price spread", r_price), ("Log spread", r_log), ("Ratio", r_ratio)]:
    apr, sh = apr_sharpe(r)
    print(f"{name:12s}  APR={apr:.3f}  Sharpe={sh:.3f}")

    #pg88


Price spread  APR=-0.014  Sharpe=-0.059
Log spread    APR=0.022  Sharpe=0.119
Ratio         APR=-0.073  Sharpe=-0.482


In [5]:
#Bollinger Band Mean Reversion Strategy

import numpy as np
import pandas as pd
import yfinance as yf

ticker = "SPY"
start = "2018-01-01"

window = 20
k = 2.0  # number of standard deviations (2 is standard)

# Entry/exit logic parameters:
# We'll do a classic mean reversion rule:
#   - Go LONG when price closes below the Lower Band (oversold)
#   - Go SHORT when price closes above the Upper Band (overbought)
#   - Exit when price returns to the moving average (the "middle band")
#

raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
price = raw["Close"]

# if this is a DataFrame (MultiIndex case), pick the single ticker column
if isinstance(price, pd.DataFrame):
    price = price[ticker]

df = price.to_frame("price").dropna()
# 2) Compute Bollinger Bands (the math)
# ------------------------------------------------------------
# Definitions:
#   Let P_t be the price at time t.
#   Middle band (moving average):
#       MA_t = mean(P_{t-window+1} ... P_t)
#   Rolling standard deviation:
#       SD_t = std(P_{t-window+1} ... P_t)
#   Upper band:
#       UB_t = MA_t + k * SD_t
#   Lower band:
#       LB_t = MA_t - k * SD_t
#
# Intuition:
#   - MA is the "typical" price over the past window.
#   - SD measures how much price usually wiggles.
#   - UB/LB define "unusually high/low" regions if price moves outside them.

df["ma"] = df["price"].rolling(window).mean()
df["sd"] = df["price"].rolling(window).std(ddof=1) #ddof means sample standard deviation

df["upper"] = df["ma"] + k * df["sd"]
df["lower"] = df["ma"] - k * df["sd"]

# Define trading signals (mean reversion logic)
# ------------------------------------------------------------
# We create a *position* series:
#   position_t = +1  means long (profit if price goes up)
#   position_t = -1  means short (profit if price goes down)
#   position_t = 0   means flat (no position)
#
# Entry rules:
#   - If price_t < lower_t  -> oversold -> go LONG (expect bounce back to MA)
#   - If price_t > upper_t  -> overbought -> go SHORT (expect drop back to MA)
#
# Exit rules:
#   - If long and price_t >= ma_t  -> exit to 0
#   - If short and price_t <= ma_t -> exit to 0
#
# IMPORTANT: We must avoid lookahead bias.
# We will generate positions based on information available at time t,
# but we will apply that position starting at t+1 when computing returns.

df["position"] = 0.0

long_entry = df["price"] < df["lower"]
short_entry = df["price"] > df["upper"]

pos = 0.0

for i in range(len(df)):
    price = df["price"].iloc[i]
    ma = df["ma"].iloc[i]

    #If MA is NAN for first window rows 
    if not np.isfinite(ma):
        df["position"].iloc[i] = 0.0
        continue
    
    #If currently flat, check for entries
    if pos == 0.0:
        if long_entry.iloc[i]:
            pos = 1.0
        elif short_entry.iloc[i]:
            pos = -1.0
    
    elif pos == 1.0:
        if price >= ma:
            pos = 0.0
    
    elif pos == -1.0:
        if price <= ma:
            pos = 0.0
    
    df["position"].iloc[i] = pos

#Calculate Returns
df["asset_ret"] = df["price"].pct_change()

# Strategy return uses yesterday's position (shift by 1):
#   strat_ret_t = position_{t-1} * asset_ret_t
#
# Why shift?
#   Because today's return happens from P_{t-1} to P_t.
#   The position you held during that period must have been decided at t-1.
df["strategy_ret"] = df["position"].shift(1) * df["asset_ret"]

df["equity"] = (1+df["strategy_ret"].fillna(0)).cumprod()

df["buy_hold"] = (1+df["asset_ret"].fillna(0)).cumprod()

ann = 252
r = df["strategy_ret"].dropna()

apr = r.mean() * ann
sharpe = np.sqrt(ann) * r.mean() / r.std(ddof=1) if r.std(ddof=1) != 0 else np.nan

# Max drawdown (common risk metric)
# Drawdown = equity / running_max - 1
running_max = df["equity"].cummax()
drawdown = df["equity"] / running_max - 1
max_dd = drawdown.min()

print("============================================================")
print(f"Bollinger Mean Reversion on {ticker}")
print("------------------------------------------------------------")
print(f"Window={window}, k={k}")
print(f"APR (approx): {apr:.3f}  -> {apr*100:.1f}%")
print(f"Sharpe:       {sharpe:.3f}")
print(f"Max Drawdown: {max_dd:.3f} -> {max_dd*100:.1f}%")
print("============================================================")

# ------------------------------------------------------------
# 6) Show a small table to sanity-check what the strategy is doing
# ------------------------------------------------------------
# Columns:
#   price: actual price
#   ma/sd/upper/lower: Bollinger bands
#   position: position decided at close
#   asset_ret: next day's price move
#   strategy_ret: PnL from holding position
#   equity: cumulative growth
print("\nLast 10 rows (sanity check):")
print(df[["price", "ma", "upper", "lower", "position", "asset_ret", "strategy_ret", "equity"]].tail(10))



C:\Users\suley\AppData\Local\Temp\ipykernel_26072\1414871522.py:84: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["position"].iloc[i] = 0.0
C:\Users\suley\AppData\Local\Temp\ipykernel_26072\1414871522.py:84: FutureWarning: ChainedAssignme

Bollinger Mean Reversion on SPY
------------------------------------------------------------
Window=20, k=2.0
APR (approx): 0.038  -> 3.8%
Sharpe:       0.239
Max Drawdown: -0.318 -> -31.8%

Last 10 rows (sanity check):
                 price          ma       upper       lower  position  \
Date                                                                   
2025-12-08  683.630005  674.897507  695.284637  654.510377       0.0   
2025-12-09  683.039978  674.977505  695.484933  654.470078       0.0   
2025-12-10  687.570007  675.206006  696.186200  654.225811       0.0   
2025-12-11  689.169983  675.495505  697.101073  653.889936       0.0   
2025-12-12  681.760010  675.981506  697.696804  654.266209       0.0   
2025-12-15  680.729980  676.421506  698.147762  654.695249       0.0   
2025-12-16  678.869995  677.081506  698.226775  655.936238       0.0   
2025-12-17  671.400024  677.647507  697.439333  657.855681       0.0   
2025-12-18  676.469971  678.339505  696.846612  659.832398  